In [ ]:
# ===== SETUP: run this first in every new session =====
import os, sys, glob, json, time, importlib, hashlib, numpy as np, pandas as pd, torch
W = "/kaggle/working"; CACHE = f"{W}/cache"; BASE = f"{W}/runs/stage3"
CONFIGS = {"baseline": dict(p_flair_only=0.0, p_post_missing=0.0),
           "seqdrop":  dict(p_flair_only=0.25, p_post_missing=0.15)}
SEEDS = [42, 43, 44]
SUBST = {"TCGA_DU_6407", "TCGA_DU_7013", "TCGA_DU_7301", "TCGA_DU_8165", "TCGA_FG_7634",
         "TCGA_FG_A60K", "TCGA_HT_7877", "TCGA_HT_8105", "TCGA_HT_A616", "TCGA_HT_A61B"}

need = {"index+split": f"{W}/lgg_index.csv", "module": f"{W}/lgg_seg.py",
        "runner": f"{W}/run_one.py", "cache": f"{CACHE}/images.npy"}
missing = [k for k, p in need.items() if not os.path.exists(p)]
if missing:
    print("Missing:", missing)
    print("-> Run in order: Cell 1, 2, 4 (if index missing), Cell 5 + 11 (if module/runner missing), "
          "Cell 6 (if cache missing). Then run this SETUP cell again.")
else:
    sys.path.insert(0, W); import lgg_seg as L; importlib.reload(L)
    assert hasattr(L, "stress_eval"), "Stage 3 patch missing - run Cell 11"
    df, imgs, msks = L.load_cache(CACHE); stats = L.patient_channel_stats(df, imgs)
    pf = df.groupby("patient_id").fold.first()
    fp = hashlib.md5("".join(f"{p}:{f};" for p, f in sorted(zip(pf.index, pf.values))).encode()).hexdigest()
    assert fp == "214d478caf1818662c0076ab66518de9", "Split changed! Stop and tell me."
    print("Ready ✔  split verified | slices:", len(df), "| GPUs:", torch.cuda.device_count())
    for n in [f"{c}_s{s}" for c in CONFIGS for s in SEEDS]:
        print(f"  {n:13s} finished: {os.path.exists(f'{BASE}/{n}/last.pt')}")

In [ ]:
# ===== Stage 1 · Cell 2: index, pairing, patient IDs, image/mask properties =====
import re, glob, hashlib, time
from collections import defaultdict
from PIL import Image

IMG_RE  = re.compile(r"^(TCGA_[A-Z0-9]+_[A-Z0-9]+)_(\d+)_(\d+)\.tif$")
MASK_RE = re.compile(r"^(TCGA_[A-Z0-9]+_[A-Z0-9]+)_(\d+)_(\d+)_mask\.tif$")

def md5(p, chunk=1 << 20):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

t0 = time.time()
all_tifs = sorted(glob.glob(os.path.join(DATA_ROOT, "**", "*.tif"), recursive=True))
print("Total .tif files under DATA_ROOT:", len(all_tifs))

# Detect duplicate copies of the dataset (several parent folders holding TCGA_* case folders)
files_by_root = defaultdict(list)
for p in all_tifs:
    files_by_root[os.path.dirname(os.path.dirname(p))].append(p)
print("Candidate roots (parent of case folders):")
for r, fl in files_by_root.items():
    print(f"  {r}: {len(fl)} tif, {len({os.path.dirname(p) for p in fl})} case folders")
roots_sorted = sorted(files_by_root, key=lambda r: (-len(files_by_root[r]), len(r)))
ROOT = roots_sorted[0]
ref = {os.path.relpath(p, ROOT) for p in files_by_root[ROOT]}
rng = np.random.default_rng(SEED)
for other in roots_sorted[1:]:
    o = {os.path.relpath(p, other) for p in files_by_root[other]}
    common = sorted(ref & o)
    sample = rng.choice(common, size=min(30, len(common)), replace=False) if common else []
    n_same = sum(md5(os.path.join(ROOT, c)) == md5(os.path.join(other, c)) for c in sample)
    print(f"  duplicate check {other}: same file list={o == ref}, md5-identical {n_same}/{len(sample)} sampled")
print("Using ROOT =", ROOT)

# Parse filenames
imgs, masks, unparsed = {}, {}, []
for p in files_by_root[ROOT]:
    fn = os.path.basename(p)
    m = MASK_RE.match(fn)
    if m: masks[(m.group(1), m.group(2), int(m.group(3)))] = p; continue
    m = IMG_RE.match(fn)
    if m: imgs[(m.group(1), m.group(2), int(m.group(3)))] = p; continue
    unparsed.append(p)
missing_mask = sorted(set(imgs) - set(masks))
orphan_mask  = sorted(set(masks) - set(imgs))
print(f"\nImages: {len(imgs)} | masks: {len(masks)} | images w/o mask: {len(missing_mask)} "
      f"| masks w/o image: {len(orphan_mask)} | unparsed files: {len(unparsed)}")
if missing_mask: print("  e.g.", missing_mask[:3])
if unparsed:     print("  e.g.", unparsed[:3])

# Load every pair
rows = []
for key in sorted(set(imgs) & set(masks)):
    pid, date, sl = key
    im = np.array(Image.open(imgs[key])); mk = np.array(Image.open(masks[key]))
    mk2 = mk[..., 0] if mk.ndim == 3 else mk
    three = im.ndim == 3 and im.shape[-1] >= 3
    rows.append(dict(
        patient_id=pid, study_date=date, slice_idx=sl,
        case_dir=os.path.basename(os.path.dirname(imgs[key])),
        image_path=imgs[key], mask_path=masks[key],
        img_shape=str(im.shape), img_dtype=str(im.dtype),
        mask_shape=str(mk.shape), mask_vals=str(tuple(np.unique(mk).tolist())),
        img_min=int(im.min()), img_max=int(im.max()),
        tumor_px=int((mk2 > 0).sum()),
        ch01_equal=bool(three and np.array_equal(im[..., 0], im[..., 1])),
        ch12_equal=bool(three and np.array_equal(im[..., 1], im[..., 2])),
        ch02_equal=bool(three and np.array_equal(im[..., 0], im[..., 2])),
    ))
df = pd.DataFrame(rows)
df["has_tumor"] = df.tumor_px > 0
df["institution"] = df.patient_id.str.split("_").str[1]
print(f"Loaded {len(df)} pairs in {time.time() - t0:.1f}s")

# Consistency checks
bad_folder = df[df.case_dir != df.patient_id + "_" + df.study_date]
print("Rows whose folder name != file prefix:", len(bad_folder))
print("Image shapes:", df.img_shape.value_counts().to_dict(), "| dtypes:", df.img_dtype.value_counts().to_dict())
print("Mask shapes:", df.mask_shape.value_counts().to_dict())
print("Mask value sets:", df.mask_vals.value_counts().to_dict())
print("Image intensity range:", df.img_min.min(), "to", df.img_max.max())

print(f"\nPatients: {df.patient_id.nunique()} | case folders: {df.case_dir.nunique()} "
      f"| max study dates per patient: {df.groupby('patient_id').study_date.nunique().max()}")
print(f"Slices: {len(df)} | tumor slices: {int(df.has_tumor.sum())} ({df.has_tumor.mean():.1%})")

pt = df.groupby("patient_id").agg(institution=("institution", "first"),
                                  n_slices=("slice_idx", "size"),
                                  n_tumor_slices=("has_tumor", "sum"),
                                  tumor_px=("tumor_px", "sum")).reset_index()
print("\nSlices per patient:\n", pt.n_slices.describe().round(1).to_string())
print("Tumor slices per patient:\n", pt.n_tumor_slices.describe().round(1).to_string())
print("Patients with zero tumor slices:", int((pt.n_tumor_slices == 0).sum()))
print("\nPer institution:\n", pt.groupby("institution").agg(patients=("patient_id", "size"),
      slices=("n_slices", "sum"), tumor_slices=("n_tumor_slices", "sum")).to_string())

noncontig = [p for p, g in df.groupby("patient_id")
             if sorted(g.slice_idx) != list(range(1, len(g) + 1))]
print("\nPatients with non-contiguous slice numbering:", len(noncontig), noncontig[:5])

print("\nTumor area (px) on positive slices:\n",
      df.loc[df.has_tumor, "tumor_px"].describe(percentiles=[.05, .25, .5, .75, .95]).round(1).to_string())
print("Positive slices with <100 tumor px:", int((df.has_tumor & (df.tumor_px < 100)).sum()))

# Channel duplication (hint of missing sequences filled with another channel)
nb = df[df.img_max > 0]
ch = nb.groupby("patient_id")[["ch01_equal", "ch12_equal", "ch02_equal"]].mean()
print("\nPatients with channels identical on >90% of non-blank slices:",
      {c: int((ch[c] > 0.9).sum()) for c in ch.columns})
print("Blank (all-zero) slices:", int((df.img_max == 0).sum()))

# Metadata files
for c in sorted(glob.glob(os.path.join(DATA_ROOT, "**", "*.csv"), recursive=True)):
    meta = pd.read_csv(c)
    print(f"\nCSV: {c} shape={meta.shape}\n  columns: {list(meta.columns)}")
    if "Patient" in meta.columns:
        s, ids = set(meta.Patient.astype(str)), set(df.patient_id)
        print(f"  patients in csv not in images: {len(s - ids)} | images not in csv: {len(ids - s)}")
for ext in ("md", "txt"):
    for f in glob.glob(os.path.join(DATA_ROOT, "**", f"*.{ext}"), recursive=True):
        print("Doc file:", f)

df.to_csv(f"{WORK_DIR}/lgg_index.csv", index=False)
print("\nSaved", f"{WORK_DIR}/lgg_index.csv")

In [ ]:
# ===== Stage 1 · Cell 1: environment + path check =====
import os, sys, platform
import numpy as np, pandas as pd, torch, sklearn, PIL

DATA_ROOT = "/kaggle/input/datasets/mateuszbuda/lgg-mri-segmentation"
WORK_DIR  = "/kaggle/working"
SEED = 42

print("Python", platform.python_version(), "| torch", torch.__version__,
      "| sklearn", sklearn.__version__, "| numpy", np.__version__, "| PIL", PIL.__version__)
print("CUDA available:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU{i}:", torch.cuda.get_device_name(i))

if not os.path.isdir(DATA_ROOT):
    print("DATA_ROOT not found. Contents of /kaggle/input (depth<=3):")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root[len("/kaggle/input"):].count(os.sep)
        print("  " * depth + os.path.basename(root) + "/", f"({len(files)} files)")
        if depth >= 3: dirs[:] = []
    raise FileNotFoundError(DATA_ROOT)

print("\nDataset tree (case folders counted, not expanded):")
for root, dirs, files in os.walk(DATA_ROOT):
    depth = root[len(DATA_ROOT):].count(os.sep)
    n_case = sum(d.startswith("TCGA_") for d in dirs)
    print("  " * depth + (os.path.basename(root) or root) + "/",
          f"| {len(dirs)} subdirs ({n_case} TCGA_*) | {len(files)} files | e.g. {sorted(files)[:3]}")
    dirs[:] = [] if depth >= 3 else sorted(d for d in dirs if not d.startswith("TCGA_"))

In [ ]:
# ===== Stage 1 · Cell 3: sample slices, all channels + GT contour =====
import matplotlib.pyplot as plt

rng = np.random.default_rng(SEED)
pos = df[df.has_tumor]
pids = rng.choice(pos.patient_id.unique(), size=6, replace=False)
picks = []
for p in pids:
    g = pos[pos.patient_id == p].sort_values("tumor_px")
    picks.append(g.iloc[len(g) // 2])          # median-area tumor slice of that patient

fig, axes = plt.subplots(6, 4, figsize=(12, 18))
for r, row in enumerate(picks):
    im = np.array(Image.open(row.image_path)); mk = np.array(Image.open(row.mask_path))
    mk = mk[..., 0] if mk.ndim == 3 else mk
    for c in range(3):
        axes[r, c].imshow(im[..., c], cmap="gray")
        axes[r, c].set_title(f"{row.patient_id} s{row.slice_idx} ch{c}", fontsize=8)
    axes[r, 3].imshow(im[..., 1], cmap="gray")
    axes[r, 3].contour(mk > 0, levels=[0.5], colors="r", linewidths=1)
    axes[r, 3].set_title(f"ch1 + GT ({row.tumor_px}px)", fontsize=8)
for a in axes.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(f"{WORK_DIR}/stage1_samples.png", dpi=110); plt.show()

In [ ]:
import base64
from IPython.display import HTML
with open("/kaggle/working/stage1_samples.png", "rb") as f:
    b64 = base64.b64encode(f.read()).decode()
HTML(f'<a download="stage1_samples.png" href="data:image/png;base64,{b64}">Download stage1_samples.png</a>')

In [ ]:
# ===== Stage 1 · Cell 4: patient-level folds (stratified by institution) =====
import json, hashlib
from sklearn.model_selection import StratifiedKFold

N_FOLDS, TEST_FOLD, VAL_FOLD = 5, 0, 1
cnt = pt.institution.value_counts()
pt["strata"] = np.where(pt.institution.map(cnt) >= N_FOLDS, pt.institution, cnt.index[0])  # merge tiny sites
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
pt["fold"] = -1
for f, (_, te) in enumerate(skf.split(pt.patient_id, pt.strata)):
    pt.loc[te, "fold"] = f
assert (pt.fold >= 0).all()
pt["split"] = np.select([pt.fold == TEST_FOLD, pt.fold == VAL_FOLD], ["test", "val"], "train")

df = df.drop(columns=["fold", "split"], errors="ignore").merge(
    pt[["patient_id", "fold", "split"]], on="patient_id", how="left", validate="many_to_one")
S = {s: set(df.loc[df.split == s, "patient_id"]) for s in ("train", "val", "test")}
assert not (S["train"] & S["val"]) and not (S["train"] & S["test"]) and not (S["val"] & S["test"])
assert sum(map(len, S.values())) == len(pt) and df.split.notna().all()
print("No patient appears in more than one split ✔")

def med_pos(x): x = x[x > 0]; return float(np.median(x)) if len(x) else 0.0
print("\nPer fold:\n", df.groupby("fold").agg(patients=("patient_id", "nunique"), slices=("slice_idx", "size"),
      tumor_slices=("has_tumor", "sum"), tumor_frac=("has_tumor", "mean"),
      median_tumor_px=("tumor_px", med_pos)).round(3).to_string())
print("\nInstitutions per fold:\n", pd.crosstab(pt.fold, pt.institution).to_string())
print("\nPreliminary split:\n", df.groupby("split").agg(patients=("patient_id", "nunique"),
      slices=("slice_idx", "size"), tumor_slices=("has_tumor", "sum")).to_string())

os.makedirs(f"{WORK_DIR}/splits", exist_ok=True)
pt.to_csv(f"{WORK_DIR}/splits/patient_folds.csv", index=False)
df.to_csv(f"{WORK_DIR}/lgg_index.csv", index=False)
fp = hashlib.md5("".join(f"{p}:{f};" for p, f in sorted(zip(pt.patient_id, pt.fold))).encode()).hexdigest()
json.dump(dict(seed=SEED, n_folds=N_FOLDS, test_fold=TEST_FOLD, val_fold=VAL_FOLD,
               stratify="institution", root=ROOT, split_fingerprint=fp,
               sklearn=sklearn.__version__), open(f"{WORK_DIR}/splits/protocol.json", "w"), indent=2)
print("\nSplit fingerprint:", fp)

In [ ]:
print(open(os.path.join(ROOT, "README.md")).read())
print(ch[(ch > 0.9).any(axis=1)].round(2).to_string())

In [ ]:
%%writefile /kaggle/working/lgg_seg.py
"""LGG FLAIR-abnormality segmentation: caching, data, U-Net, losses, metrics, training, evaluation, plots."""
import os, json, time, random
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage
from torch.utils.data import Dataset, DataLoader
from torchvision import tv_tensors
from torchvision.transforms import v2, InterpolationMode

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

# ---------------- cache (read the .tif files once) ----------------
def _read_pair(paths):
    ip, mp = paths
    im = np.array(Image.open(ip)); mk = np.array(Image.open(mp))
    mk = mk[..., 0] if mk.ndim == 3 else mk
    return im, (mk > 0).astype(np.uint8)

def build_cache(index_csv, cache_dir, n_threads=16):
    os.makedirs(cache_dir, exist_ok=True)
    f_img, f_msk, f_idx = (os.path.join(cache_dir, n) for n in ("images.npy", "masks.npy", "index.csv"))
    if all(os.path.exists(f) for f in (f_img, f_msk, f_idx)):
        print("Cache already exists:", cache_dir); return
    df = pd.read_csv(index_csv).sort_values(["patient_id", "slice_idx"]).reset_index(drop=True)
    assert "split" in df.columns, "Run Cell 4 first (index must contain the split)."
    imgs = np.empty((len(df), 256, 256, 3), np.uint8); msks = np.empty((len(df), 256, 256), np.uint8)
    t0 = time.time()
    with ThreadPoolExecutor(n_threads) as ex:
        for i, (im, mk) in enumerate(ex.map(_read_pair, zip(df.image_path, df.mask_path))):
            imgs[i], msks[i] = im, mk
    np.save(f_img, imgs); np.save(f_msk, msks); df.to_csv(f_idx, index=False)
    print(f"Cached {len(df)} slices in {time.time() - t0:.0f}s -> {cache_dir}")

def load_cache(cache_dir):
    df = pd.read_csv(os.path.join(cache_dir, "index.csv"))
    return df, np.load(os.path.join(cache_dir, "images.npy")), np.load(os.path.join(cache_dir, "masks.npy"))

def patient_channel_stats(df, imgs):
    """Per-patient, per-channel mean/std (uses only that patient's own volume -> no cross-split leakage)."""
    stats = np.zeros((len(df), 2, 3), np.float32)
    for _, idx in df.groupby("patient_id").indices.items():
        v = imgs[idx].reshape(-1, 3).astype(np.float32)
        stats[idx, 0] = v.mean(0); stats[idx, 1] = v.std(0) + 1e-6
    return stats

# ---------------- dataset ----------------
class SliceDataset(Dataset):
    def __init__(self, imgs, msks, stats, indices, augment=False):
        self.imgs, self.msks, self.stats, self.idx = imgs, msks, stats, np.asarray(indices)
        self.tf = v2.Compose([
            v2.RandomHorizontalFlip(0.5),
            v2.RandomAffine(degrees=15, scale=(0.9, 1.1), interpolation=InterpolationMode.BILINEAR),
        ]) if augment else None
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        j = int(self.idx[i])
        x = torch.from_numpy(self.imgs[j]).permute(2, 0, 1).float()
        m = torch.from_numpy(self.msks[j])[None]
        if self.tf is not None:                       # joint transform; masks use nearest interpolation
            x, m = self.tf(tv_tensors.Image(x), tv_tensors.Mask(m))
            x, m = x.as_subclass(torch.Tensor), m.as_subclass(torch.Tensor)
        mean = torch.from_numpy(self.stats[j, 0]).view(3, 1, 1)
        std = torch.from_numpy(self.stats[j, 1]).view(3, 1, 1)
        return (x - mean) / std, m.float(), j

# ---------------- model ----------------
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        c = [base * 2 ** i for i in range(5)]
        self.enc = nn.ModuleList([conv_block(in_ch, c[0])] + [conv_block(c[i], c[i + 1]) for i in range(3)])
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = conv_block(c[3], c[4])
        self.up = nn.ModuleList([nn.ConvTranspose2d(c[i + 1], c[i], 2, stride=2) for i in reversed(range(4))])
        self.dec = nn.ModuleList([conv_block(2 * c[i], c[i]) for i in reversed(range(4))])
        self.head = nn.Conv2d(c[0], out_ch, 1)
    def forward(self, x):
        skips = []
        for e in self.enc:
            x = e(x); skips.append(x); x = self.pool(x)
        x = self.bottleneck(x)
        for up, dec, s in zip(self.up, self.dec, reversed(skips)):
            x = dec(torch.cat([up(x), s], 1))
        return self.head(x)

# ---------------- losses ----------------
def soft_dice_loss(logits, target, eps=1.0):
    p = torch.sigmoid(logits)
    inter = (p * target).sum()
    return 1 - (2 * inter + eps) / (p.sum() + target.sum() + eps)   # batch-level, safe for empty slices

def make_loss(name):
    bce = nn.BCEWithLogitsLoss()
    if name == "bce": return bce
    if name == "bce_dice": return lambda lg, t: bce(lg, t) + soft_dice_loss(lg, t)
    raise ValueError(name)

# ---------------- metrics ----------------
STRUCT_6 = ndimage.generate_binary_structure(3, 1)   # 6-connectivity, as in Paper 1

def largest_cc_3d(vol):
    lab, n = ndimage.label(vol, structure=STRUCT_6)
    if n <= 1: return vol
    sizes = ndimage.sum(vol, lab, range(1, n + 1))
    return lab == (int(np.argmax(sizes)) + 1)

def _counts(p, g):
    return int(np.count_nonzero(p & g)), int(np.count_nonzero(p & ~g)), int(np.count_nonzero(~p & g))

def _overlap(tp, fp, fn):
    den = 2 * tp + fp + fn
    return dict(dice=2 * tp / den if den else 1.0,
                iou=tp / (tp + fp + fn) if den else 1.0,
                precision=tp / (tp + fp) if (tp + fp) else float("nan"),
                sensitivity=tp / (tp + fn) if (tp + fn) else float("nan"))

def evaluate(df, msks, probs, indices, thr=0.5, postprocess=False, n_boot=2000, seed=0):
    """Per-patient 3D metrics (primary), pooled Dice, slice-level Dice on tumor slices, FP on tumor-free slices."""
    idx = np.asarray(indices); pred_all = probs >= thr
    sub = df.iloc[idx][["patient_id", "slice_idx"]].copy(); sub["_pos"] = np.arange(len(idx))
    prows, srows = [], []
    for pid, g in sub.groupby("patient_id", sort=True):
        g = g.sort_values("slice_idx"); pos = g["_pos"].to_numpy()
        pv = pred_all[pos]; gv = msks[idx[pos]].astype(bool)
        if postprocess: pv = largest_cc_3d(pv)
        tp, fp, fn = _counts(pv, gv)
        prows.append(dict(patient_id=pid, n_slices=len(pos), gt_voxels=int(gv.sum()), pred_voxels=int(pv.sum()),
                          tp=tp, fp=fp, fn=fn, **_overlap(tp, fp, fn)))
        for k, s in enumerate(g["slice_idx"].to_numpy()):
            t, f_, n_ = _counts(pv[k], gv[k])
            srows.append(dict(patient_id=pid, slice_idx=int(s), row=int(idx[pos[k]]), gt_px=int(gv[k].sum()),
                              pred_px=int(pv[k].sum()), tp=t, fp=f_, fn=n_, dice=_overlap(t, f_, n_)["dice"]))
    pat, sl = pd.DataFrame(prows), pd.DataFrame(srows)
    pos_s, neg_s = sl[sl.gt_px > 0], sl[sl.gt_px == 0]
    T, Fp, Fn = (int(pat[c].sum()) for c in ("tp", "fp", "fn"))
    d = pat.dice.to_numpy(); rng = np.random.default_rng(seed)
    boots = [rng.choice(d, len(d), replace=True).mean() for _ in range(n_boot)]
    summary = dict(
        postprocess=postprocess, threshold=thr, n_patients=len(pat),
        patient_dice_mean=float(d.mean()), patient_dice_median=float(np.median(d)),
        patient_dice_std=float(d.std(ddof=1)) if len(d) > 1 else float("nan"),
        patient_dice_ci95=[round(float(np.percentile(boots, 2.5)), 4), round(float(np.percentile(boots, 97.5)), 4)],
        patient_iou_mean=float(pat.iou.mean()), patient_precision_mean=float(np.nanmean(pat.precision)),
        patient_sensitivity_mean=float(pat.sensitivity.mean()), pooled_dice=_overlap(T, Fp, Fn)["dice"],
        n_tumor_slices=len(pos_s), tumor_slice_dice_mean=float(pos_s.dice.mean()),
        fn_slices_no_prediction=int((pos_s.pred_px == 0).sum()),
        n_empty_slices=len(neg_s), fp_slices=int((neg_s.pred_px > 0).sum()),
        fp_slice_rate=float((neg_s.pred_px > 0).mean()) if len(neg_s) else float("nan"),
        fp_px_total_on_empty=int(neg_s.pred_px.sum()))
    return summary, pat, sl

# ---------------- inference / training ----------------
@torch.no_grad()
def predict_probs(model, imgs, msks, stats, indices, device, batch_size=32, num_workers=2):
    model.eval()
    dl = DataLoader(SliceDataset(imgs, msks, stats, indices, False), batch_size=batch_size,
                    shuffle=False, num_workers=num_workers, pin_memory=True)
    out = np.empty((len(indices), 256, 256), np.float16); k = 0
    for x, _, _ in dl:
        with torch.autocast("cuda", dtype=torch.float16):
            p = torch.sigmoid(model(x.to(device, non_blocking=True)).float())
        out[k:k + len(x)] = p[:, 0].cpu().numpy(); k += len(x)
    return out

def train_model(cfg, df, imgs, msks, stats, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    json.dump(cfg, open(os.path.join(out_dir, "config.json"), "w"), indent=2)
    seed_everything(cfg["seed"]); torch.backends.cudnn.benchmark = True
    device = torch.device(cfg["device"])
    tr = np.where(df.split == "train")[0]; va = np.where(df.split == "val")[0]
    gen = torch.Generator(); gen.manual_seed(cfg["seed"])
    dl = DataLoader(SliceDataset(imgs, msks, stats, tr, True), batch_size=cfg["batch_size"], shuffle=True,
                    num_workers=cfg["num_workers"], drop_last=True, generator=gen, pin_memory=True,
                    persistent_workers=cfg["num_workers"] > 0)
    model = UNet(base=cfg["base"]).to(device)
    print(f"Params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M | train slices {len(tr)} | val slices {len(va)}")
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])
    scaler = torch.amp.GradScaler("cuda"); loss_fn = make_loss(cfg["loss"])
    best, wait, hist = -1.0, 0, []
    for ep in range(1, cfg["epochs"] + 1):
        model.train(); t0 = time.time(); tot = n = 0
        for x, m, _ in dl:
            x, m = x.to(device, non_blocking=True), m.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(x)
            loss = loss_fn(logits.float(), m)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * len(x); n += len(x)
        lr = opt.param_groups[0]["lr"]; sched.step(); t_tr = time.time() - t0
        t1 = time.time()
        summ, _, _ = evaluate(df, msks, predict_probs(model, imgs, msks, stats, va, device), va, n_boot=0 or 1)
        t_va = time.time() - t1; vd = summ["patient_dice_mean"]
        hist.append(dict(epoch=ep, lr=lr, train_loss=tot / n, val_patient_dice=vd,
                         val_pooled_dice=summ["pooled_dice"], val_fp_slice_rate=summ["fp_slice_rate"],
                         t_train_s=t_tr, t_val_s=t_va))
        pd.DataFrame(hist).to_csv(os.path.join(out_dir, "history.csv"), index=False)
        flag = ""
        if vd > best:
            best, wait, flag = vd, 0, " *best*"
            torch.save(dict(model=model.state_dict(), epoch=ep, val_dice=float(vd), cfg=cfg),
                       os.path.join(out_dir, "best.pt"))
        else:
            wait += 1
        print(f"ep {ep:3d} | loss {tot / n:.4f} | val pDice {vd:.4f} | val FP-slice {summ['fp_slice_rate']:.3f} "
              f"| {t_tr:.0f}s train + {t_va:.0f}s val{flag}")
        if wait >= cfg["patience"]:
            print(f"Early stop at epoch {ep} (no val improvement for {cfg['patience']} epochs)"); break
    torch.save(dict(model=model.state_dict(), epoch=ep, val_dice=float(vd), cfg=cfg), os.path.join(out_dir, "last.pt"))
    print(f"Best val patient Dice {best:.4f}")
    return pd.DataFrame(hist)

def load_model(run_dir, device, which="best.pt"):
    ck = torch.load(os.path.join(run_dir, which), map_location=device)
    m = UNet(base=ck["cfg"]["base"]).to(device); m.load_state_dict(ck["model"]); m.eval()
    print(f"Loaded {which}: epoch {ck['epoch']}, val patient Dice {ck['val_dice']:.4f}")
    return m

# ---------------- plots ----------------
def plot_history(run_dir):
    h = pd.read_csv(os.path.join(run_dir, "history.csv"))
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    ax[0].plot(h.epoch, h.train_loss); ax[0].set_title("train loss"); ax[0].set_xlabel("epoch")
    ax[1].plot(h.epoch, h.val_patient_dice, label="val patient Dice")
    ax[1].plot(h.epoch, h.val_fp_slice_rate, label="val FP-slice rate"); ax[1].legend(); ax[1].set_xlabel("epoch")
    plt.tight_layout(); plt.savefig(os.path.join(run_dir, "history.png"), dpi=110); plt.show()

def _norm01(a):
    a = a.astype(np.float32); return (a - a.min()) / (a.max() - a.min() + 1e-6)

def _err_rgb(base, pred, gt, alpha=0.55):
    rgb = np.repeat(base[..., None], 3, -1)
    for m, c in ((pred & gt, (0, 1, 0)), (pred & ~gt, (1, 0, 0)), (~pred & gt, (0, 0.45, 1))):
        rgb[m] = (1 - alpha) * rgb[m] + alpha * np.array(c)
    return rgb

def plot_cases(df, imgs, msks, probs, indices, pat, sl, out_png, thr=0.5, n_fp=3):
    idx = np.asarray(indices); rpos = {int(r): i for i, r in enumerate(idx)}
    ps = pat.sort_values("dice").reset_index(drop=True)
    rows = []
    for tag, p in (("worst", ps.iloc[0]), ("median", ps.iloc[len(ps) // 2]), ("best", ps.iloc[-1])):
        s = sl[sl.patient_id == p.patient_id].sort_values("gt_px").iloc[-1]
        rows.append((f"{tag}: {p.patient_id} (patient Dice {p.dice:.3f}) s{s.slice_idx}", int(s.row)))
    for _, s in sl[(sl.gt_px == 0) & (sl.pred_px > 0)].sort_values("pred_px", ascending=False).head(n_fp).iterrows():
        rows.append((f"FP, tumor-free slice: {s.patient_id} s{s.slice_idx} ({s.pred_px}px)", int(s.row)))
    fig, ax = plt.subplots(len(rows), 4, figsize=(13, 3.3 * len(rows)), squeeze=False)
    for r, (title, row) in enumerate(rows):
        base = _norm01(imgs[row][..., 1]); gt = msks[row].astype(bool)
        pr = probs[rpos[row]].astype(np.float32); pb = pr >= thr
        ax[r, 0].imshow(base, cmap="gray"); ax[r, 0].set_title(title, fontsize=8, loc="left")
        ax[r, 1].imshow(base, cmap="gray"); ax[r, 1].set_title("FLAIR + GT", fontsize=8)
        if gt.any(): ax[r, 1].contour(gt, [0.5], colors="lime", linewidths=1)
        ax[r, 2].imshow(pr, cmap="magma", vmin=0, vmax=1); ax[r, 2].set_title("probability", fontsize=8)
        ax[r, 3].imshow(_err_rgb(base, pb, gt)); ax[r, 3].set_title("TP green / FP red / FN blue", fontsize=8)
    for a in ax.ravel(): a.axis("off")
    plt.tight_layout(); plt.savefig(out_png, dpi=100); plt.show()

In [ ]:
import sys, importlib, numpy as np, pandas as pd, torch
sys.path.insert(0, "/kaggle/working")
import lgg_seg as L; importlib.reload(L)

CACHE = "/kaggle/working/cache"
L.build_cache("/kaggle/working/lgg_index.csv", CACHE)
df, imgs, msks = L.load_cache(CACHE)
stats = L.patient_channel_stats(df, imgs)
assert (df.groupby("patient_id").split.nunique() == 1).all()
print(imgs.shape, imgs.dtype, msks.shape, "mask max", msks.max())
print(df.groupby("split").agg(patients=("patient_id", "nunique"), slices=("slice_idx", "size")).to_string())

In [ ]:
CFG = dict(seed=42, base=32, batch_size=16, lr=3e-4, wd=1e-4, epochs=50, patience=12,
           loss="bce", num_workers=4, device="cuda:0")
_ = L.train_model({**CFG, "epochs": 2}, df, imgs, msks, stats, "/kaggle/working/runs/timing")

In [ ]:
RUN = "/kaggle/working/runs/baseline_bce_s42"
hist = L.train_model(CFG, df, imgs, msks, stats, RUN)
L.plot_history(RUN)

In [ ]:
import json
device = torch.device("cuda:0")
model = L.load_model(RUN, device)
te = np.where(df.split == "test")[0]
probs_te = L.predict_probs(model, imgs, msks, stats, te, device)

res = {}
for pp in (False, True):
    summ, pat, sl = L.evaluate(df, msks, probs_te, te, postprocess=pp)
    res["largest_3D_CC" if pp else "raw"] = summ
    pat.to_csv(f"{RUN}/test_patients_{'cc' if pp else 'raw'}.csv", index=False)
    if not pp: pat_raw, sl_raw = pat, sl
print(pd.DataFrame(res).to_string())
json.dump(res, open(f"{RUN}/test_summary.json", "w"), indent=2)

SUBST = {"TCGA_DU_6407", "TCGA_DU_7013", "TCGA_DU_7301", "TCGA_DU_8165", "TCGA_FG_7634",
         "TCGA_FG_A60K", "TCGA_HT_7877", "TCGA_HT_8105", "TCGA_HT_A616", "TCGA_HT_A61B"}
s = pat_raw[pat_raw.patient_id.isin(SUBST)]
print(f"\nSubstituted-channel patients in test: {len(s)}",
      f"| their mean Dice {s.dice.mean():.4f}" if len(s) else "")
print("\nPer-patient (raw), sorted by Dice:\n", pat_raw.sort_values("dice")[
    ["patient_id", "n_slices", "gt_voxels", "pred_voxels", "dice", "precision", "sensitivity"]].round(3).to_string(index=False))
L.plot_cases(df, imgs, msks, probs_te, te, pat_raw, sl_raw, f"{RUN}/test_cases.png")

In [ ]:
import base64, os
from IPython.display import HTML, display
RUN = "/kaggle/working/runs/baseline_bce_s42"
links = []
for name in ("history.png", "test_cases.png"):
    p = os.path.join(RUN, name)
    if not os.path.exists(p):
        print("Missing:", p); continue
    b64 = base64.b64encode(open(p, "rb").read()).decode()
    links.append(f'<a download="{name}" href="data:image/png;base64,{b64}">Download {name}</a>')
display(HTML("<br>".join(links)))

In [ ]:
print(df[df.patient_id.isin(SUBST)].groupby("split").patient_id.unique().to_dict())

In [ ]:
device = torch.device("cuda:0")
model = L.load_model(RUN, device)

# (a) Validation patients with substituted channels (incl. FLAIR-only HT_A616)
va = np.where(df.split == "val")[0]
summ_va, pat_va, _ = L.evaluate(df, msks, L.predict_probs(model, imgs, msks, stats, va, device), va)
print(f"Val mean patient Dice: {summ_va['patient_dice_mean']:.4f}")
print(pat_va[pat_va.patient_id.isin(SUBST)][["patient_id", "dice", "precision", "sensitivity"]].round(3).to_string(index=False))

# (b) Stress test: full-sequence test patients, with sequences replaced by FLAIR
te_full = np.where((df.split == "test") & ~df.patient_id.isin(SUBST))[0]
df_t = df.iloc[te_full].reset_index(drop=True)
base_imgs, m_t, ar = imgs[te_full], msks[te_full], np.arange(len(te_full))
scenarios = {"original": (), "post_missing (ch2=FLAIR)": (2,),
             "pre_missing (ch0=FLAIR)": (0,), "FLAIR_only (ch0=ch2=FLAIR)": (0, 2)}
rows, per = [], {}
for name, chans in scenarios.items():
    x = base_imgs.copy()
    for c in chans: x[..., c] = x[..., 1]
    st = L.patient_channel_stats(df_t, x)          # renormalise exactly as for real substituted patients
    pr = L.predict_probs(model, x, m_t, st, ar, device)
    s, p, _ = L.evaluate(df_t, m_t, pr, ar)
    per[name] = p.set_index("patient_id").dice
    rows.append(dict(scenario=name, n_pat=s["n_patients"], pDice_mean=s["patient_dice_mean"],
                     pDice_median=s["patient_dice_median"], ci95=s["patient_dice_ci95"],
                     pooled=s["pooled_dice"], prec=s["patient_precision_mean"],
                     sens=s["patient_sensitivity_mean"], fp_slices=s["fp_slices"],
                     fn_slices=s["fn_slices_no_prediction"]))
print("\n", pd.DataFrame(rows).round(4).to_string(index=False))

d = pd.DataFrame(per); delta = d.sub(d["original"], axis=0)
print("\nPer-patient Dice change vs original:")
for c in delta.columns[1:]:
    print(f"  {c}: mean {delta[c].mean():+.4f} | worst {delta[c].min():+.4f} | "
          f"patients dropping >0.10: {int((delta[c] < -0.10).sum())}/{len(delta)}")
print("\nPer-patient Dice by scenario:\n", d.round(3).to_string())
d.round(4).to_csv(f"{RUN}/stress_test_per_patient.csv")

In [ ]:
import importlib, torch
path = "/kaggle/working/lgg_seg.py"
EXT = '''

# ---------------- Stage 3: training-time sequence dropout ----------------
class SeqDropDataset(SliceDataset):
    # Replaces pre-/post-contrast with FLAIR, following the dataset's own convention for missing sequences.
    # Channels are normalised per patient, so copying normalised FLAIR equals renormalising a substituted volume.
    def __init__(self, imgs, msks, stats, indices, augment=True, p_flair_only=0.0, p_post_missing=0.0):
        super().__init__(imgs, msks, stats, indices, augment)
        self.p1, self.p2 = p_flair_only, p_post_missing
    def __getitem__(self, i):
        x, m, j = super().__getitem__(i)
        u = torch.rand(()).item()
        if u < self.p1:
            x[0] = x[1]; x[2] = x[1]
        elif u < self.p1 + self.p2:
            x[2] = x[1]
        return x, m, j

def make_train_dataset(cfg, imgs, msks, stats, tr):
    p1, p2 = cfg.get("p_flair_only", 0.0), cfg.get("p_post_missing", 0.0)
    if p1 + p2 > 0:
        return SeqDropDataset(imgs, msks, stats, tr, True, p_flair_only=p1, p_post_missing=p2)
    return SliceDataset(imgs, msks, stats, tr, True)

SCENARIOS = {"original": (), "post_missing": (2,), "pre_missing": (0,), "flair_only": (0, 2)}

def stress_eval(model, df, imgs, msks, rows, device, scenarios=SCENARIOS):
    df_t = df.iloc[rows].reset_index(drop=True); m_t = msks[rows]; ar = np.arange(len(rows))
    summ, per = {}, {}
    for name, chans in scenarios.items():
        x = imgs[rows].copy()
        for c in chans:
            x[..., c] = x[..., 1]
        st = patient_channel_stats(df_t, x)
        s, p, _ = evaluate(df_t, m_t, predict_probs(model, x, m_t, st, ar, device), ar, n_boot=1)
        summ[name] = s; per[name] = p.set_index("patient_id").dice
    return summ, pd.DataFrame(per)
'''
src = open(path).read()
old = "dl = DataLoader(SliceDataset(imgs, msks, stats, tr, True),"
new = "dl = DataLoader(make_train_dataset(cfg, imgs, msks, stats, tr),"
if "make_train_dataset" not in src:
    assert src.count(old) == 1, "Module text differs from expected - stop and tell me."
    open(path, "w").write(src.replace(old, new) + EXT)
importlib.reload(L)

# Sanity checks
tr_idx = np.where(df.split == "train")[0][:4]
x, _, _ = L.SeqDropDataset(imgs, msks, stats, tr_idx, True, p_flair_only=1.0)[0]
assert torch.equal(x[0], x[1]) and torch.equal(x[2], x[1])
x, _, _ = L.SeqDropDataset(imgs, msks, stats, tr_idx, True, p_post_missing=1.0)[0]
assert torch.equal(x[2], x[1]) and not torch.equal(x[0], x[1])
x, _, _ = L.SeqDropDataset(imgs, msks, stats, tr_idx, False, p_flair_only=1.0)[0]   # no aug -> still applies? check below
print("Sequence-dropout sanity checks passed")

open("/kaggle/working/run_one.py", "w").write('''import sys, os, json
sys.path.insert(0, "/kaggle/working")
import lgg_seg as L
cfg, out = json.loads(sys.argv[1]), sys.argv[2]
if os.path.exists(os.path.join(out, "last.pt")):
    print("Already finished:", out); sys.exit(0)
df, imgs, msks = L.load_cache("/kaggle/working/cache")
L.train_model(cfg, df, imgs, msks, L.patient_channel_stats(df, imgs), out)
''')
print("Runner written.")

In [ ]:
import json, shlex, subprocess, os
try:
    del model
except NameError:
    pass
torch.cuda.empty_cache()

BASE = "/kaggle/working/runs/stage3"; os.makedirs(BASE, exist_ok=True)
COMMON = dict(base=32, batch_size=16, lr=3e-4, wd=1e-4, epochs=50, patience=12,
              loss="bce", num_workers=2, device="cuda:0")
CONFIGS = {"baseline": dict(p_flair_only=0.0, p_post_missing=0.0),
           "seqdrop":  dict(p_flair_only=0.25, p_post_missing=0.15)}
SEEDS = [42, 43, 44]
job = lambda c, s: (f"{c}_s{s}", {**COMMON, **CONFIGS[c], "seed": s})
queues = {0: [job("baseline", 42), job("seqdrop", 43), job("baseline", 44)],
          1: [job("seqdrop", 42), job("baseline", 43), job("seqdrop", 44)]}
procs = {}
for gpu, q in queues.items():
    cmds = [f"python -u /kaggle/working/run_one.py {shlex.quote(json.dumps(cfg))} {BASE}/{name} > {BASE}/{name}.log 2>&1"
            for name, cfg in q]
    procs[gpu] = subprocess.Popen(["bash", "-c", " ; ".join(cmds)],
                                  env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}, start_new_session=True)
    print(f"GPU{gpu}: {[n for n, _ in q]} (pid {procs[gpu].pid})")

In [ ]:
import glob
for log in sorted(glob.glob(f"{BASE}/*.log")):
    lines = open(log).read().strip().splitlines()
    print(f"{os.path.basename(log):20s} | {lines[-1] if lines else '(starting)'}")
print({f"GPU{g}": ("running" if p.poll() is None else f"done (exit {p.returncode})") for g, p in procs.items()})

In [ ]:
   print(open(f"{BASE}/baseline_s42.log").read())
   print(open(f"{BASE}/seqdrop_s42.log").read())

In [ ]:
from scipy.stats import wilcoxon
device = torch.device("cuda:0")
te_all = np.where(df.split == "test")[0]
te_full = np.where((df.split == "test") & ~df.patient_id.isin(SUBST))[0]
records, per = [], {}
for c in CONFIGS:
    for s in SEEDS:
        d = f"{BASE}/{c}_s{s}"
        model = L.load_model(d, device)
        s22, p22 = L.stress_eval(model, df, imgs, msks, te_all, device, {"original": ()})
        s21, p21 = L.stress_eval(model, df, imgs, msks, te_full, device)
        rec = dict(config=c, seed=s, val_best=pd.read_csv(f"{d}/history.csv").val_patient_dice.max(),
                   test22_original=s22["original"]["patient_dice_mean"],
                   FG_A60K=p22.loc["TCGA_FG_A60K", "original"])
        for sc, sm in s21.items():
            rec[f"full21_{sc}"] = sm["patient_dice_mean"]
        rec["full21_flair_only_sens"] = s21["flair_only"]["patient_sensitivity_mean"]
        rec["full21_original_fp_slices"] = s21["original"]["fp_slices"]
        records.append(rec); per[(c, s)] = p21
        del model; torch.cuda.empty_cache()

R = pd.DataFrame(records)
print(R.round(4).to_string(index=False))
print("\nMean ± SD over 3 seeds:\n", R.drop(columns="seed").groupby("config").agg(["mean", "std"]).round(4).T.to_string())

print("\nPaired per-patient comparison (21 full-sequence test patients, Dice averaged over seeds):")
avg = {c: sum(per[(c, s)] for s in SEEDS) / len(SEEDS) for c in CONFIGS}
for sc in L.SCENARIOS:
    diff = avg["seqdrop"][sc] - avg["baseline"][sc]
    print(f"  {sc:13s} baseline {avg['baseline'][sc].mean():.4f} | seqdrop {avg['seqdrop'][sc].mean():.4f} "
          f"| diff {diff.mean():+.4f} | seqdrop better {int((diff > 0).sum())}/{len(diff)} "
          f"| Wilcoxon p={wilcoxon(diff).pvalue:.4f}")
R.to_csv(f"{BASE}/stage3_runs.csv", index=False)
pd.concat({f"{c}_s{s}": p for (c, s), p in per.items()}, axis=1).round(4).to_csv(f"{BASE}/stage3_per_patient.csv")